In [17]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
import joblib

MODEL_DIR = Path("models/random-forest")

In [18]:
df = pd.read_csv("Data/clean_fifa_worldcup_matches.csv")

In [19]:
print(df.head())


    HomeTeam AwayTeam  Year  HomeGoals  AwayGoals  TotalGoals
0     France   Mexico  1930          4          1           5
1  Argentina   France  1930          1          0           1
2      Chile   Mexico  1930          3          0           3
3      Chile   France  1930          1          0           1
4  Argentina   Mexico  1930          6          3           9


In [20]:
required_columns = {'HomeTeam', 'AwayTeam', 'Year', 'HomeGoals', 'AwayGoals', 'TotalGoals'}
if not required_columns.issubset(df.columns):
    raise ValueError("Dataset is missing required columns!")

In [21]:
all_teams = pd.concat([df['HomeTeam'], df['AwayTeam']]).unique()


In [22]:
team_encoder = LabelEncoder()
team_encoder.fit(all_teams)


LabelEncoder()

In [23]:
df['HomeTeamEncoded'] = team_encoder.transform(df['HomeTeam'])
df['AwayTeamEncoded'] = team_encoder.transform(df['AwayTeam'])

In [24]:
X = df[['HomeTeamEncoded', 'AwayTeamEncoded', 'Year']]
y_home = df['HomeGoals']
y_away = df['AwayGoals']

In [25]:
# Single split so home/away targets stay aligned on the same rows
X_train, X_test, y_train_home, y_test_home, y_train_away, y_test_away = train_test_split(
    X, y_home, y_away, test_size=0.2, random_state=42
)

home_model = RandomForestRegressor(n_estimators=100, random_state=42)
away_model = RandomForestRegressor(n_estimators=100, random_state=42)

home_model.fit(X_train, y_train_home)
away_model.fit(X_train, y_train_away)

RandomForestRegressor(random_state=42)

In [26]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(home_model, MODEL_DIR / "home_goal_model.pkl")
joblib.dump(away_model, MODEL_DIR / "away_goal_model.pkl")
joblib.dump(team_encoder, MODEL_DIR / "team_encoder.pkl")
print(f"Models trained and saved to {MODEL_DIR}/")


Models trained and saved to models/random-forest/


In [27]:
# Load saved artifacts so this section can run without retraining
team_encoder = joblib.load(MODEL_DIR / "team_encoder.pkl")
fixtures_2026 = pd.read_csv("Data/clean_fifa_worldcup_fixture.csv")


In [28]:
def encode_team(team):
    if team in team_encoder.classes_:
        return team_encoder.transform([team])[0]
    else:
        return -1 

In [29]:
# Fixture CSV uses lowercase column names: home, away, year
fixtures_2026.rename(
    columns={'home': 'HomeTeam', 'away': 'AwayTeam', 'year': 'Year'},
    inplace=True,
)

fixtures_2026['HomeTeamEncoded'] = fixtures_2026['HomeTeam'].apply(encode_team)
fixtures_2026['AwayTeamEncoded'] = fixtures_2026['AwayTeam'].apply(encode_team)


In [30]:
# Drop matches where either team never appeared in historical data
fixtures_2026 = fixtures_2026[
    (fixtures_2026['HomeTeamEncoded'] != -1) & (fixtures_2026['AwayTeamEncoded'] != -1)
].copy()

X_2026 = fixtures_2026[['HomeTeamEncoded', 'AwayTeamEncoded']].copy()
X_2026.loc[:, 'Year'] = 2026

home_model = joblib.load(MODEL_DIR / "home_goal_model.pkl")
away_model = joblib.load(MODEL_DIR / "away_goal_model.pkl")

In [31]:
fixtures_2026['PredictedHomeGoals'] = home_model.predict(X_2026)
fixtures_2026['PredictedAwayGoals'] = away_model.predict(X_2026)

# Round predictions to nearest whole number
fixtures_2026['PredictedHomeGoals'] = fixtures_2026['PredictedHomeGoals'].round().astype(int)
fixtures_2026['PredictedAwayGoals'] = fixtures_2026['PredictedAwayGoals'].round().astype(int)

# Add a score column
fixtures_2026['Score'] = fixtures_2026['PredictedHomeGoals'].astype(str) + " - " + fixtures_2026['PredictedAwayGoals'].astype(str)

In [32]:
fixtures_2026[['HomeTeam', 'Score', 'AwayTeam', 'Year']].to_csv("Data/fifa_worldcup_2026_Score_predictions.csv", index=False)

print("Predictions for World Cup 2026 saved successfully!")

Predictions for World Cup 2026 saved successfully!
